In [7]:
import json
import os
import uuid
from datetime import datetime
from abc import ABC, abstractmethod

# ==========================================
# 1. ADVANCED CONCEPTS: DECORATORS
# ==========================================

def auto_save(func):
    """
    Decorator: Automatically calls self.save_data() after the function executes.
    This eliminates the need to write self.save_data() manually inside every method.
    """
    def wrapper(self, *args, **kwargs):
        result = func(self, *args, **kwargs)
        self.save_data()
        return result
    return wrapper


# ==========================================
# 2. CUSTOM EXCEPTIONS (Inheritance)
# ==========================================

class HotelSystemException(Exception): pass
class InvalidInputException(HotelSystemException): pass
class RoomConflictException(HotelSystemException): pass
class RoomNotFoundException(HotelSystemException): pass
class BookingException(HotelSystemException): pass


# ==========================================
# 3. DATA MODELS: ABSTRACTION, ENCAPSULATION, POLYMORPHISM
# ==========================================

class BaseRoom(ABC):
    """Abstract Base Class: Defines the blueprint for all room types."""
    def __init__(self, room_number, room_type, price, is_available=True):
        self.room_number = str(room_number).strip()
        self.room_type = str(room_type).strip().capitalize()
        self.__price = float(price)  # Private variable (Encapsulation)
        self.is_available = bool(is_available)

    # Getter for price (Encapsulation)
    @property
    def price(self):
        return self.__price

    # Setter for price with validation
    @price.setter
    def price(self, value):
        if value <= 0:
            raise InvalidInputException("Price must be greater than zero.")
        self.__price = float(value)

    # Abstract Method: Forces child classes to create their own billing rules
    @abstractmethod
    def calculate_bill(self, days):
        pass

    def to_dict(self):
        return {
            "room_number": self.room_number,
            "room_type": self.room_type,
            "price": self.price,
            "is_available": self.is_available
        }

    # Magic Method: Controls how the room prints
    def __str__(self):
        return f"Room: {self.room_number.ljust(5)} | Type: {self.room_type.ljust(8)} | Price: INR {self.price}/night"


class StandardRoom(BaseRoom):
    """Child Class (Inheritance). Used for Single and Double rooms."""
    def calculate_bill(self, days):
        # Polymorphism: Standard billing
        return self.price * days


class SuiteRoom(BaseRoom):
    """Child Class (Inheritance). Used for Suite rooms."""
    def calculate_bill(self, days):
        # Polymorphism: Suites have a mandatory flat luxury tax added
        luxury_tax = 1000.0
        return (self.price * days) + luxury_tax


class RoomFactory:
    """Design Pattern: Factory Pattern. Instantiates the correct child class based on data."""
    @staticmethod
    def create(room_number, room_type, price, is_available=True):
        if str(room_type).strip().capitalize() == "Suite":
            return SuiteRoom(room_number, room_type, price, is_available)
        else:
            return StandardRoom(room_number, room_type, price, is_available)


class Booking:
    def __init__(self, guest_name, room_number, duration_days, booking_id=None, check_in_date=None):
        self.booking_id = booking_id or str(uuid.uuid4())[:8].upper()
        self.guest_name = str(guest_name).strip()
        self.room_number = str(room_number).strip()
        self.duration_days = int(duration_days)
        self.check_in_date = check_in_date or datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def to_dict(self):
        return {
            "booking_id": self.booking_id,
            "guest_name": self.guest_name,
            "room_number": self.room_number,
            "duration_days": self.duration_days,
            "check_in_date": self.check_in_date
        }


# ==========================================
# 4. CORE CONTROLLER
# ==========================================

class HotelManager:
    VALID_ROOM_TYPES = ["Single", "Double", "Suite"]

    def __init__(self, data_file="hotel_data.json"):
        self.data_file = data_file
        self.rooms = {}      # Format: { "101": RoomObject }
        self.bookings = {}   # Format: { "101": BookingObject }
        self.load_data()

    def load_data(self):
        if not os.path.exists(self.data_file):
            return

        try:
            with open(self.data_file, 'r') as file:
                data = json.load(file)
                
                # Unpack rooms using the Factory Pattern
                for r_data in data.get("rooms", []):
                    room_obj = RoomFactory.create(**r_data)
                    self.rooms[r_data["room_number"]] = room_obj
                    
                for b_data in data.get("bookings", []):
                    self.bookings[b_data["room_number"]] = Booking(**b_data)
                    
        except json.JSONDecodeError:
            print("[CRITICAL] Database file is corrupted. Starting empty.")
        except Exception as e:
            print(f"[CRITICAL] Load error: {e}")

    def save_data(self):
        try:
            data = {
                "rooms": [room.to_dict() for room in self.rooms.values()],
                "bookings": [booking.to_dict() for booking in self.bookings.values()]
            }
            with open(self.data_file, 'w') as file:
                json.dump(data, file, indent=4)
        except Exception as e:
            print(f"[CRITICAL] Save error: {e}")

    @auto_save  # Uses the Decorator we defined at the top
    def add_room(self, room_number, room_type, price):
        room_number = str(room_number).strip()
        room_type = str(room_type).strip().capitalize()

        if not room_number or not room_type:
            raise InvalidInputException("Room number and room type cannot be blank.")
        if room_number in self.rooms:
            raise RoomConflictException(f"Room {room_number} already exists.")
        if room_type not in self.VALID_ROOM_TYPES:
            raise InvalidInputException(f"Invalid type. Approved: {', '.join(self.VALID_ROOM_TYPES)}")

        # Create room via Factory
        self.rooms[room_number] = RoomFactory.create(room_number, room_type, price)
        print(f"[SUCCESS] Room {room_number} ({room_type}) recorded.")

    def _generate_available_rooms(self):
        """Generator: Yields available rooms one by one efficiently."""
        for room in self.rooms.values():
            if room.is_available:
                yield room

    def display_available_rooms(self):
        print("\n--- Available Rooms ---")
        
        # Consuming the Generator
        available = list(self._generate_available_rooms())
        
        if not available:
            print("No rooms are currently available in the system.")
            return
        
        for room in available:
            print(room)  # This automatically triggers the __str__ Magic Method
        print("-----------------------")

    @auto_save # Uses decorator
    def check_in(self, guest_name, room_number, duration_days):
        guest_name = str(guest_name).strip()
        room_number = str(room_number).strip()

        # Input Validation
        if not guest_name:
            raise InvalidInputException("Guest name cannot be blank.")
        if any(char.isdigit() for char in guest_name):
            raise InvalidInputException("Guest name cannot contain numbers. Please use alphabetic characters only.")
        if room_number not in self.rooms:
            raise RoomNotFoundException(f"Room {room_number} does not exist.")
        if not self.rooms[room_number].is_available:
            raise BookingException(f"Room {room_number} is occupied.")
        if duration_days <= 0:
            raise InvalidInputException("Stay must be at least 1 night.")

        room = self.rooms[room_number]
        
        # Use polymorphism to calculate expected bill based on room type
        expected_bill = room.calculate_bill(duration_days)

        booking = Booking(guest_name, room_number, duration_days)
        self.bookings[room_number] = booking
        room.is_available = False
        
        print(f"[SUCCESS] Checked in. Booking ID: {booking.booking_id}")
        print(f"          Estimated Bill ({duration_days} nights): INR {expected_bill}")

    @auto_save # Uses decorator
    def check_out(self, room_number):
        room_number = str(room_number).strip()

        if room_number not in self.bookings:
            raise BookingException(f"No active booking found for Room {room_number}.")

        booking = self.bookings.pop(room_number)
        room = self.rooms[room_number]
        room.is_available = True
        
        # Polymorphism in action: Suites apply tax, Standard rooms don't
        total_bill = room.calculate_bill(booking.duration_days)
        
        print("\n=== INVOICE ===")
        print(f"Booking ID  : {booking.booking_id}")
        print(f"Guest Name  : {booking.guest_name}")
        print(f"Room Number : {booking.room_number} ({room.room_type})")
        print(f"Check-In    : {booking.check_in_date}")
        print(f"Duration    : {booking.duration_days} nights")
        print(f"Total Bill  : INR {total_bill}")
        print("===============\n")


# ==========================================
# 5. USER INTERFACE LAYER (CLI)
# ==========================================

def main():
    hotel = HotelManager()

    while True:
        print("\n==============================")
        print("   HOTEL MANAGEMENT SYSTEM")
        print("==============================")
        print("1. Add a New Room")
        print("2. View Available Rooms")
        print("3. Check-In Guest")
        print("4. Check-Out & Generate Bill")
        print("5. Exit")
        print("==============================")

        choice = input("Enter your choice (1-5): ").strip()

        try:
            if choice == '1':
                r_num = input("Enter Room Number: ").strip()
                if r_num in hotel.rooms:
                    print(f"[ERROR] Room {r_num} already exists!")
                    continue
                    
                r_type = input("Enter Room Type (Single/Double/Suite): ").strip()
                
                try:
                    price = float(input("Enter Price per Night (INR): ").strip())
                except ValueError:
                    print("[ERROR] Pricing issue. Use numeric values.")
                    continue

                hotel.add_room(r_num, r_type, price)

            elif choice == '2':
                hotel.display_available_rooms()

            elif choice == '3':
                name = input("Enter Guest Name: ").strip()
                r_num = input("Enter Room Number: ").strip()
                
                try:
                    duration = int(input("Enter Stay Duration (in days): ").strip())
                except ValueError:
                    print("[ERROR] Duration issue. Input whole integers.")
                    continue

                hotel.check_in(name, r_num, duration)

            elif choice == '4':
                r_num = input("Enter Room Number to Check-out: ").strip()
                hotel.check_out(r_num)

            elif choice == '5':
                print("\nShutting down safely. Goodbye.")
                break
                
            else:
                print("[ERROR] Pick a choice from 1 to 5.")

        except HotelSystemException as business_error:
            print(f"[BUSINESS LOGIC ERROR] {business_error}")
        except Exception as e:
            print(f"[SYSTEM ERROR] {e}")


if __name__ == "__main__":
    main()


   HOTEL MANAGEMENT SYSTEM
1. Add a New Room
2. View Available Rooms
3. Check-In Guest
4. Check-Out & Generate Bill
5. Exit


Enter your choice (1-5):  1
Enter Room Number:  107


[ERROR] Room 107 already exists!

   HOTEL MANAGEMENT SYSTEM
1. Add a New Room
2. View Available Rooms
3. Check-In Guest
4. Check-Out & Generate Bill
5. Exit


Enter your choice (1-5):  1
Enter Room Number:  105


[ERROR] Room 105 already exists!

   HOTEL MANAGEMENT SYSTEM
1. Add a New Room
2. View Available Rooms
3. Check-In Guest
4. Check-Out & Generate Bill
5. Exit


...

[BUSINESS LOGIC ERROR] Room 105 is occupied.

   HOTEL MANAGEMENT SYSTEM
1. Add a New Room
2. View Available Rooms
3. Check-In Guest
4. Check-Out & Generate Bill
5. Exit


Enter your choice (1-5):  3
Enter Guest Name:  abc
Enter Room Number:  104
Enter Stay Duration (in days):  5


[BUSINESS LOGIC ERROR] Room 104 does not exist.

   HOTEL MANAGEMENT SYSTEM
1. Add a New Room
2. View Available Rooms
3. Check-In Guest
4. Check-Out & Generate Bill
5. Exit


Enter your choice (1-5):  5



Shutting down safely. Goodbye.
